# Evaluation in LangChain

- When building an application with LLM, it is important to evaluate how well the application is working
- Check if changes in design helps improve the application results or not
- Evaluation can be done manually (with LangChain debug), assisted with LLM or LangChain evaluation platform

### _EXAMPLE:_

#### _Evaluate the Question Answering over Documents:_

Import these -
- RetrivalQA from chain which will do retrival over documents
- Language model
- CVS loader from document loaders
- Vector store index creator to create a vectore store
- Doc array in memory search from vector stores which is an in memory vector store and does not need to connect to external database - there are many types of vector stores

    from langchain.chains import RetrievalQA
    from langchain.chat_models import ChatOpenAI
    from langchain.document_loaders import CSVLoader
    from langchain.indexes import VectorstoreIndexCreator
    from langchain.vectorstores import DocArrayInMemorySearch


Load Example csv file:

    file = 'OutdoorClothingCatalog_1000.csv'
    loader = CSVLoader(file_path=file)
    data = loader.load()

Create an index - vector database for the document:

_Specify vector store class as docs array in memory search and take in list of documents to be loaded :_

    index = VectorstoreIndexCreator(
        vectorstore_cls=DocArrayInMemorySearch
    ).from_loaders([loader])

Specify the Q&A Chain with language model, chain type, retriver:

    llm = ChatOpenAI(temperature = 0.0, model=llm_model)
    qa = RetrievalQA.from_chain_type(
        llm=llm, 
        chain_type="stuff", 
        retriever=index.vectorstore.as_retriever(), 
        verbose=True,
        chain_type_kwargs = {
            "document_separator": "<<<<>>>>>"
        }
    )

#### _Manual Example Questions:_

_Questions based on the data in document._

    examples = [
        {
            "query": "Do the Cozy Comfort Pullover Set\
            have side pockets?",
            "answer": "Yes"
        },
        {
            "query": "What collection is the Ultra-Lofty \
            850 Stretch Down Hooded Jacket from?",
            "answer": "The DownTek collection"
        }
    ]

#### _LLM Generated Examples Questions:_

Import QA Generate Chain to take documents to create question answer pair from each document:

    from langchain.evaluation.qa import QAGenerateChain

    example_gen_chain = QAGenerateChain.from_llm(ChatOpenAI(model=llm_model))

    new_examples = example_gen_chain.apply_and_parse(
        [{"doc": t} for t in data[:5]]
    )

Combine Hand-made and LLM generated examples:

    examples += new_examples

Run qa chain with a query:

    qa.run(examples[0]["query"])

#### _Manual Evaluation with LangChain Debug :_

Use LangChain debug to get more information on the process for running the query. It shows the retrival qa, stuffs, llm chain and llm model steps in detail. It also shows details about the return message process.

Set Langchain debug to true:

    import langchain
    langchain.debug = True

Run qa chain with a query:

    qa.run(examples[0]["query"])

#### _LLM Assisted Evaluation:_

Use LLM to make predictions for all example queries:

    predictions = qa.apply(examples)

#### _QA Evaluate Chain:_

Import QA Eval Chain to Evaluate:

    from langchain.evaluation.qa import QAEvalChain

    llm = ChatOpenAI(temperature=0, model=llm_model)
    eval_chain = QAEvalChain.from_llm(llm)

Evaluate the predictions with the example answers:

    graded_outputs = eval_chain.evaluate(examples, predictions)

    for i, eg in enumerate(examples):
        print(f"Example {i}:")
        print("Question: " + predictions[i]['query'])
        print("Real Answer: " + predictions[i]['answer'])
        print("Predicted Answer: " + predictions[i]['result'])
        print("Predicted Grade: " + graded_outputs[i]['text'])
        print()